# Predicting Restaurant Health Code Failures: A Discrete-Time Survival Analysis

**Author:** Sarayu Rao

This notebook models *when* a Seattle restaurant first receives an unsatisfactory
health inspection result, using a discrete-time (event history) approach. Each
routine inspection a restaurant undergoes is treated as one period at risk of
"failure," and I ask what predicts the hazard of failing in a given period.

Data source: [Seattle Food Establishment Inspection Data](https://data.seattle.gov/)
(public open data portal).


In [ ]:
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt


## 1. Load and prepare the data

I define **failure** as a restaurant receiving an *Unsatisfactory* inspection
result. Not every violation point results in an unsatisfactory rating, and a
restaurant can accumulate violation points across several inspections without
ever failing outright, so this is a genuine hazard question rather than a
simple count model.

I restrict to **routine inspections in Seattle**, since these happen on a
regular cadence and are comparable across restaurants. Complaint-driven or
follow-up inspections don't reflect the same "at risk" process.


In [ ]:
data = pd.read_csv("data/Food_Establishment_Inspection_Data.csv")

# Binary failure indicator: 1 = Unsatisfactory result
data["inspection_fail"] = (data["Inspection Result"] == "Unsatisfactory").astype(int)

# Parse dates and build a monthly period index
data["Inspection Date"] = pd.to_datetime(data["Inspection Date"])
data["month_year"] = data["Inspection Date"].dt.to_period("M")

# Global month ticker (1, 2, 3, ...) so month can be used as a numeric covariate
unique_months = sorted(data["month_year"].unique())
months = {month: i + 1 for i, month in enumerate(unique_months)}
data["month"] = data["month_year"].map(months)

# Keep only routine inspections in Seattle
routine_data = data[
    (data["Inspection Type"] == "Routine Inspection/Field Review")
    & (data["City"] == "SEATTLE")
]

routine_data.head()


## 2. State the failure variable and expected effects

My dataset includes food safety inspections for restaurants in Seattle. My
failure variable is when a restaurant receives its first unsatisfactory
rating — meaning it received a critical violation that could increase the
risk of foodborne illness.

I predict that the number of violation points a restaurant receives in an
inspection will affect when it receives its first unsatisfactory result. Not
all violation points result in an unsatisfactory rating, and a restaurant can
receive multiple violation points but still avoid an unsatisfactory rating,
but I still expect an effect.


In [ ]:
routine_data["inspection_fail"].value_counts(normalize=True).sort_index() * 100


This shows that, out of all routine inspections in the data, just over half
are actually unsatisfactory.

## 3. Build the survival dataset

**Step 1** — Keep the columns needed for the model and collapse to one row per
restaurant/month, summing violation points within that period. Then identify
and remove restaurants whose *first* observation is already a failure
(left-censored — I can't observe their true time at risk before they enter
the data).

**Step 2** — Flag `first_1_inspection_fail`, the period in which a restaurant
first receives an unsatisfactory result. Periods *after* that first failure
are set to `None` rather than dropped outright — the restaurant has exited
the risk pool for a "first" failure, and these `None` rows are excluded later
via `.dropna()` when building the correlation matrix and logit model.

**Step 3** — Add `time_since_entry`, a within-restaurant counter of how many
periods it has been at risk.


In [ ]:
# Step 1: Filter relevant columns and remove restaurants whose first observation is already a failure
filtered_data = routine_data[
    ["Name", "month", "month_year", "City", "Violation Points", "inspection_fail"]
].dropna(subset=["inspection_fail"])
filtered_data = filtered_data.groupby(
    ["Name", "month", "month_year", "City", "inspection_fail"], as_index=False
).agg({"Violation Points": "sum"})

first_observation_1 = filtered_data.groupby("Name").first()
inspections_to_remove = first_observation_1[first_observation_1["inspection_fail"] == 1].index
filtered_data = filtered_data[~filtered_data["Name"].isin(inspections_to_remove)]

# Step 2: Add "first_1_inspection_fail" to indicate the first period a restaurant gets an
# unsatisfactory result. Periods after that first failure are set to None (the restaurant
# is no longer "at risk" for a first failure) and are excluded downstream via .dropna().
filtered_data["first_1_inspection_fail"] = 0

def assign_first_1_indicator(inspection_data):
    if (inspection_data["inspection_fail"] == 1).any():
        first_1_year = inspection_data.loc[inspection_data["inspection_fail"] == 1, "month_year"].min()
        inspection_data.loc[inspection_data["month_year"] == first_1_year, "first_1_inspection_fail"] = 1
        inspection_data.loc[inspection_data["month_year"] > first_1_year, "first_1_inspection_fail"] = None
    return inspection_data

filtered_data = filtered_data.groupby("Name", group_keys=False)[filtered_data.columns].apply(
    assign_first_1_indicator
)

# Step 3: Add a "time_since_entry" variable that increments from 1 onward for each restaurant
def add_time_since_entry(inspection_data):
    inspection_data = inspection_data.sort_values(by="month_year").reset_index(drop=True)
    inspection_data["time_since_entry"] = range(1, len(inspection_data) + 1)
    return inspection_data

filtered_data = filtered_data.groupby("Name", group_keys=False)[filtered_data.columns].apply(
    add_time_since_entry
)

filtered_data.head()


## 4. Defining the risk window (left-censoring and right-truncation)

I remove any restaurant whose first observation is already a failure, since
there may be factors from before the data window (or before a restaurant even
opened, if it's new) that drove that result — and I have no way to observe
those. Also, once a restaurant receives its first unsatisfactory result, I no
longer track it for *additional* failures. My main goal is to see when a
restaurant first gets an unsatisfactory result, not how many times it fails
overall.


In [ ]:
# Plotting the outcome variable 'first_1_inspection_fail' by year to observe its distribution over time

# Group data by year and calculate the mean of 'first_1_inspection_fail' for each year
filtered_data["year"] = filtered_data["month_year"].dt.year
yearly_outcome = filtered_data.groupby("year")["first_1_inspection_fail"].mean()
all_years = range(filtered_data["year"].min(), filtered_data["year"].max() + 1)

# Plotting
plt.figure(figsize=(15, 6))
plt.plot(yearly_outcome.index, yearly_outcome.values, marker="o", linestyle="-")
plt.title("Average Rate of First Unsatisfactory Result by Year")
plt.xlabel("Year")
plt.ylabel("Share of at-risk restaurants failing for the first time")
plt.xticks(all_years, rotation=45, ha="right")
plt.grid(True)
plt.show()


For easier readability, I grouped the inspections by year for this plot, but
I still measure time in months in the model itself.


In [ ]:
# Step 1: Identify restaurants that experienced the outcome at their first observation and remove them
filtered_data2 = data[["Name", "month_year", "inspection_fail"]].dropna(subset=["inspection_fail"])
first_observation_1 = filtered_data2.groupby("Name").first()
restaurants_removed = first_observation_1[first_observation_1["inspection_fail"] == 1].index.tolist()

# Step 2: Identify restaurants that did not experience the outcome at their first observation
restaurants_not_removed = first_observation_1[first_observation_1["inspection_fail"] != 1].index.tolist()

# Step 3: Split the restaurants that didn't experience the outcome at first observation
# into those that eventually experienced the outcome and those that never did
filtered_data2 = filtered_data2[filtered_data2["Name"].isin(restaurants_not_removed)]
experienced_outcome = filtered_data2[filtered_data2["inspection_fail"] == 1]["Name"].unique().tolist()
never_experienced_outcome = [
    restaurant for restaurant in restaurants_not_removed if restaurant not in experienced_outcome
]

# Step 4: Calculate counts and prepare lists for output
num_restaurants_removed = len(restaurants_removed)
num_experienced = len(experienced_outcome)
num_never_experienced = len(never_experienced_outcome)

# Display the results
(num_restaurants_removed, num_experienced, num_never_experienced)


Looking at the graph, there's a spike in first-unsatisfactory ratings in
2007. I'm not sure why, other than maybe that once the data became public,
inspectors were especially more critical of restaurants. There's also a
decent dip in 2019-2020 — 2020 plausibly reflects reduced inspection activity
during COVID, though I'm not sure what happened in 2019.

Looking at the restaurants, 2,769 were excluded for having their first
inspection be unsatisfactory (left-censored). It's also interesting that only
about 25% of all restaurants never received an unsatisfactory result.

## 5. Multiple-variable survival analysis

I ran this using discrete-time methods (event history analysis). First, a
correlation matrix for the variables of interest.


In [ ]:
# Calculating the correlation matrix for the relevant variables in the dataset
correlation_data = filtered_data[
    ["first_1_inspection_fail", "Violation Points", "time_since_entry", "month"]
].dropna()

correlation_matrix = correlation_data.corr()
correlation_matrix


In [ ]:
# Ensure the required data is present
logit_data = filtered_data.dropna(
    subset=["first_1_inspection_fail", "Violation Points", "month_year", "month", "time_since_entry"]
)
logit_data = logit_data.rename(columns={"Violation Points": "violation_points"})

# Run logistic regression using formula style
logit_model_formula = smf.logit(
    formula="first_1_inspection_fail ~ violation_points + time_since_entry + month",
    data=logit_data,
)
logit_result_formula = logit_model_formula.fit()

# Output summary of logistic regression
logit_result_formula_summary = logit_result_formula.summary()
logit_result_formula_summary


## 6. Interpretation

The main coefficient of interest is `violation_points`, which is **0.3884**.
This means that for every increase in violation points received, the log-odds
of getting an unsatisfactory result increases by 0.3884 — a fairly large
increase in likelihood, and the p-value is significant. This makes sense:
the more violation points a restaurant has, the more issues it has that
could cause a more severe/unsatisfactory result.

The `time_since_entry` coefficient is **0.0719** and significant, meaning the
likelihood of getting an unsatisfactory result slightly increases the longer
a restaurant continues to receive routine inspections.

I also included `month` to control for a general trend over time, but that
coefficient is not significant — over the years, there hasn't been a
meaningful shift in the overall rate of unsatisfactory ratings.

The pseudo R-squared is **0.5383**, much higher than I expected. This
suggests violation points and time-since-entry do a decent job explaining
when a restaurant gets its first unsatisfactory rating. One thing that would
be interesting to explore next is whether a restaurant's *prior* violation
points (not just the current inspection's) affect whether it gets its first
unsatisfactory rating.
